In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/project-phase-1/data/sc_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/test.json
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_train.jsonl
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_val.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tribunals_val_f.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tribunals_train_f.bin
/kaggle/input/datasets/achive

In [ ]:
import json, torch
R = "/kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real"
rows = [json.loads(l) for l in open(f"{R}/log.jsonl")]
print(len(rows), "evaluations | last record:", rows[-1])
for n in ["best", "latest"]:
    ck = torch.load(f"{R}/{n}.pt", map_location="cpu")
    print(n, "-> step", ck["step"], "| best-so-far val mean", round(ck["best"], 4))

20 evaluations | last record: {'step': 10000, 'tokens': 163840000, 'train_loss': 4.202952494621277, 'val_mean': 4.100625859101613, 'lr': 3.000000738169851e-05, 'grad_norm': 1.0020785331726074, 'tok_per_sec': 12994.864968149948, 'acts_val': 3.926503096818924, 'sc_val': 4.275461745262146, 'tribunals_val_f': 4.0999127352237705}
best -> step 10000 | best-so-far val mean 4.1006


# Generation + Submission Packaging — Architecture 1

Self-contained notebook: includes the model code, so no other cell is needed.

**Inputs attached to the notebook:**
- `phase1-tokens-v40k-clean1` (tokenizer.json, test.json)
- `project-phase-1` (raw val .jsonl, for `MODE="val"`)
- `train-model-1` notebook output (`run_arch1_real/best.pt`)

**Modes:**
- `MODE = "test"`: generate for official test prompts → `prediction.json` + zip
- `MODE = "val"`: generate for held-out validation pairs → `val_pred.json` / `val_ref.json`, for scoring with the COURSE evaluator, so you can compare decoding settings before spending submissions.

In [4]:
import os, json, time, math, random, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

## Model (your architecture, unchanged)

In [5]:
# Published GPT-2 reference value. Confirm against your lecture; the choice is yours.
INIT_STD = 0.02


def _init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size, block_size, d_model, n_layers, n_heads, d_ff):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, block_size)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight      # tied embeddings

        self.apply(_init_weights)                        # CHANGE 1: small-std init

    def forward(self, x):
        B, T = x.shape
        assert T <= self.block_size

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        h = tok + pos

        for block in self.blocks:
            h = block(h)

        h = self.final_norm(h)
        return self.lm_head(h)


def build_model(vocab_size, block_size):
    d_model = 768
    n_layers = 8
    n_heads = 8
    d_ff = 3072

    return LegalDecoderLM(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        d_ff=d_ff,
    )

    ## Settings

**Decoding settings:** `TEMP=1.0 / TOP_K=0 / TOP_P=1.0` is the UNMODIFIED model distribution (no decoding intervention): a neutral starting point for the timing test, NOT a recommendation. Choosing better settings is your experiment: change these, run `MODE="val"`, score with the course evaluator, compare.

- `TEMP=0` → greedy
- `TOP_K=0` → off
- `TOP_P=1.0` → off

In [6]:
MODE = "test"                  # "test" or "val"
RANGE = (0, 5)                 # test mode: (0, 5) = timing test; (0, None) = all 1000
N_VAL_PER_SOURCE = 20          # val mode: pairs per source (acts / sc / tribunals)

CKPT = "/kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt"
TOKD = "/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens"
RAW  = "/kaggle/input/datasets/achiver7/project-phase-1/data"
OUT  = "/kaggle/working/submission_arch1"
V, BLOCK, dev = 40000, 1024, "cuda"
os.makedirs(OUT, exist_ok=True)

GEN = dict(TEMP=1.0, TOP_K=0, TOP_P=1.0, SEED=0)
GEN_TAG = f"t{GEN['TEMP']}_k{GEN['TOP_K']}_p{GEN['TOP_P']}_s{GEN['SEED']}"

assert torch.cuda.is_available(), "turn on the GPU"
assert os.path.exists(CKPT), f"missing checkpoint: {CKPT}"

## Load model & tokenizer

In [7]:
tok = Tokenizer.from_file(f"{TOKD}/tokenizer.json")
test = json.load(open(f"{TOKD}/test.json"))

model = build_model(V, BLOCK).to(dev)
ck = torch.load(CKPT, map_location=dev)
model.load_state_dict(ck["model"])
model.eval()
print(f"loaded {CKPT} | step {ck.get('step')} | best val {ck.get('best')}")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

slim = f"{OUT}/model_weights_only.pt"
if not os.path.exists(slim):
    torch.save({"model": ck["model"], "step": ck.get("step")}, slim)
    print("saved weights-only copy:", slim)
del ck

gen = torch.Generator(device=dev).manual_seed(GEN["SEED"])

loaded /kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt | step 10000 | best val 4.100625859101613
parameters: 88,210,944
saved weights-only copy: /kaggle/working/submission_arch1/model_weights_only.pt


## Decoding

In [8]:
def sample(lg):
    if GEN["TEMP"] == 0:
        return lg.argmax(-1, keepdim=True)
    lg = lg / GEN["TEMP"]
    if GEN["TOP_K"]:
        v, _ = torch.topk(lg, min(GEN["TOP_K"], lg.size(-1)))
        lg[lg < v[:, [-1]]] = -float("inf")
    p = F.softmax(lg, -1)
    if GEN["TOP_P"] < 1.0:
        sp, si = torch.sort(p, descending=True)
        cum = sp.cumsum(-1)
        sp[(cum - sp) > GEN["TOP_P"]] = 0
        p = torch.zeros_like(p).scatter_(1, si, sp)
        p = p / p.sum(-1, keepdim=True)
    return torch.multinomial(p, 1, generator=gen)


@torch.no_grad()
def continue_text(prompt, max_new=600):
    x = torch.tensor([tok.encode(prompt).ids[-BLOCK:]], device=dev)
    new = []
    while len(new) < max_new:
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x[:, -BLOCK:])[:, -1, :].float()
        nxt = sample(lg)
        x = torch.cat([x, nxt], 1)[:, -BLOCK:]
        new.append(nxt.item())
        if len(new) % 8 == 0 and len(tok.decode(new).split()) >= 128:
            break
    return tok.decode(new).split()      # new tokens only: the prompt is never included


def run_generation(prompts, tag):
    """prompts: {id: prompt_text}. Resumable via a part file keyed by tag + decoding settings."""
    part = f"{OUT}/pred_{tag}_{GEN_TAG}.json"
    pred = json.load(open(part)) if os.path.exists(part) else {}
    t0, todo = time.time(), [k for k in prompts if k not in pred]
    for n, k in enumerate(todo):
        words = continue_text(prompts[k])
        if len(words) < 128:
            raise RuntimeError(f"ID {k}: only {len(words)} words generated")
        pred[k] = {"text": " ".join(words[:128])}
        if (n + 1) % 25 == 0 or n + 1 == len(todo):
            json.dump(pred, open(part, "w"))
            print(f"{len(pred)}/{len(prompts)} done | {(time.time()-t0)/(n+1):.1f}s per prompt")
    json.dump(pred, open(part, "w"))
    assert all(len(v["text"].split()) == 128 for v in pred.values()), "not exactly 128 words"
    return pred

## Run

In [9]:
if MODE == "test":
    ids_all = list(test.keys())
    a, b = RANGE[0], (RANGE[1] if RANGE[1] is not None else len(ids_all))
    prompts = {k: test[k]["text"] for k in ids_all[a:b]}
    pred = run_generation(prompts, f"test_{a}_{b}")
    assert not any(test[k]["text"][:60] in v["text"] for k, v in pred.items()), "prompt leaked"
    print(f"checked {len(pred)} predictions: 128 words each, no prompt text")

    if set(pred) == set(test):
        with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
            json.dump(pred, f, ensure_ascii=False)
        with zipfile.ZipFile(f"{OUT}/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        print("zip root:", zipfile.ZipFile(f"{OUT}/submission.zip").namelist(), "| IDs:", len(pred))
    else:
        print(f"PARTIAL ({len(pred)}/{len(test)} IDs): no zip written. To merge parts from several")
        print("accounts: merged = {}; [merged.update(json.load(open(p))) for p in part_files];")
        print("then assert set(merged) == set(test), write prediction.json, zip with it at the root.")

elif MODE == "val":
    rng, pairs = random.Random(1), []          # fixed seed: same pairs for every decoding setting
    for s in ["acts_val", "sc_val", "tribunals_val"]:
        c = 0
        with open(f"{RAW}/{s}.jsonl", encoding="utf-8") as f:
            for line in f:
                t = json.loads(line)["text"]
                w = t.split()
                if len(w) < 640 or any(ord(ch) > 127 for ch in t):   # test prompts are pure ASCII
                    continue
                i = rng.randint(0, len(w) - 640)
                pairs.append((s, " ".join(w[i:i + 512]), " ".join(w[i + 512:i + 640])))
                c += 1
                if c >= N_VAL_PER_SOURCE:
                    break
    prompts = {str(i): p for i, (_, p, _) in enumerate(pairs)}
    pred = run_generation(prompts, f"val_{N_VAL_PER_SOURCE}")
    refs = {str(i): {"text": r} for i, (_, _, r) in enumerate(pairs)}
    json.dump(pred, open(f"{OUT}/val_pred_{GEN_TAG}.json", "w"))
    json.dump(refs, open(f"{OUT}/val_ref.json", "w"))
    print(f"wrote val_pred_{GEN_TAG}.json and val_ref.json ({len(pairs)} pairs)")
    try:
        from rouge_score import rouge_scorer
        sc_ = rouge_scorer.RougeScorer(["rougeL"])
        for s in ["acts_val", "sc_val", "tribunals_val"]:
            ix = [i for i, (n, _, _) in enumerate(pairs) if n == s]
            if ix:
                r = np.mean([sc_.score(pairs[i][2], pred[str(i)]["text"])["rougeL"].fmeasure for i in ix])
                print(f"ROUGE-L F1 {s}: {r:.4f} (n={len(ix)})")
    except ImportError:
        print("rouge_score not installed: score val_pred / val_ref with the course evaluator instead.")
    print("Score BERTScore F1 / ROUGE-L / BLEU-4 with the COURSE evaluator for the official numbers.")
else:
    raise ValueError("MODE must be 'test' or 'val'")

5/5 done | 2.7s per prompt
checked 5 predictions: 128 words each, no prompt text
PARTIAL (5/1000 IDs): no zip written. To merge parts from several
accounts: merged = {}; [merged.update(json.load(open(p))) for p in part_files];
then assert set(merged) == set(test), write prediction.json, zip with it at the root.


In [10]:
# ============================================================
# GENERATION + SUBMISSION PACKAGING - ARCHITECTURE 1 (self-contained)
#
# Contains your own model code (copied unchanged from the training file),
# so no other cell is needed. Inputs attached to the notebook:
#   - phase1-tokens-v40k-clean1   (tokenizer.json, test.json)
#   - project-phase-1             (raw val .jsonl, for MODE="val")
#   - train-model-1 notebook output (run_arch1_real/best.pt)
#
# MODE = "test": generate for official test prompts -> prediction.json + zip
# MODE = "val" : generate for held-out validation pairs -> val_pred.json /
#                val_ref.json, for scoring with the COURSE evaluator, so you
#                can compare decoding settings before spending submissions.
# ============================================================
import os, json, time, math, random, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

# ============================================================
# MODEL (your architecture, unchanged)
# ============================================================
# Published GPT-2 reference value. Confirm against your lecture; the choice is yours.
INIT_STD = 0.02


def _init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size, block_size, d_model, n_layers, n_heads, d_ff):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, block_size)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight      # tied embeddings

        self.apply(_init_weights)                        # CHANGE 1: small-std init

    def forward(self, x):
        B, T = x.shape
        assert T <= self.block_size

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        h = tok + pos

        for block in self.blocks:
            h = block(h)

        h = self.final_norm(h)
        return self.lm_head(h)


def build_model(vocab_size, block_size):
    d_model = 768
    n_layers = 8
    n_heads = 8
    d_ff = 3072

    return LegalDecoderLM(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        d_ff=d_ff,
    )



# ============================================================
# SETTINGS
# ============================================================
MODE = "test"                  # "test" or "val"
RANGE = (0, 5)                 # test mode: (0, 5) = timing test; (0, None) = all 1000
N_VAL_PER_SOURCE = 20          # val mode: pairs per source (acts / sc / tribunals)

CKPT = "/kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt"
TOKD = "/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens"
RAW  = "/kaggle/input/datasets/achiver7/project-phase-1/data"
OUT  = "/kaggle/working/submission_arch1"
V, BLOCK, dev = 40000, 1024, "cuda"
os.makedirs(OUT, exist_ok=True)

# Decoding settings. TEMP=1.0 / TOP_K=0 / TOP_P=1.0 is the UNMODIFIED model
# distribution (no decoding intervention): a neutral starting point for the
# timing test, NOT a recommendation. Choosing better settings is your experiment:
# change these, run MODE="val", score with the course evaluator, compare.
# TEMP=0 -> greedy. TOP_K=0 -> off. TOP_P=1.0 -> off.
GEN = dict(TEMP=1.0, TOP_K=0, TOP_P=1.0, SEED=0)
GEN_TAG = f"t{GEN['TEMP']}_k{GEN['TOP_K']}_p{GEN['TOP_P']}_s{GEN['SEED']}"

assert torch.cuda.is_available(), "turn on the GPU"
assert os.path.exists(CKPT), f"missing checkpoint: {CKPT}"

# ============================================================
# LOAD
# ============================================================
tok = Tokenizer.from_file(f"{TOKD}/tokenizer.json")
test = json.load(open(f"{TOKD}/test.json"))

model = build_model(V, BLOCK).to(dev)
ck = torch.load(CKPT, map_location=dev)
model.load_state_dict(ck["model"])
model.eval()
print(f"loaded {CKPT} | step {ck.get('step')} | best val {ck.get('best')}")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

slim = f"{OUT}/model_weights_only.pt"
if not os.path.exists(slim):
    torch.save({"model": ck["model"], "step": ck.get("step")}, slim)
    print("saved weights-only copy:", slim)
del ck

gen = torch.Generator(device=dev).manual_seed(GEN["SEED"])

# ============================================================
# DECODING
# ============================================================
def sample(lg):
    if GEN["TEMP"] == 0:
        return lg.argmax(-1, keepdim=True)
    lg = lg / GEN["TEMP"]
    if GEN["TOP_K"]:
        v, _ = torch.topk(lg, min(GEN["TOP_K"], lg.size(-1)))
        lg[lg < v[:, [-1]]] = -float("inf")
    p = F.softmax(lg, -1)
    if GEN["TOP_P"] < 1.0:
        sp, si = torch.sort(p, descending=True)
        cum = sp.cumsum(-1)
        sp[(cum - sp) > GEN["TOP_P"]] = 0
        p = torch.zeros_like(p).scatter_(1, si, sp)
        p = p / p.sum(-1, keepdim=True)
    return torch.multinomial(p, 1, generator=gen)


@torch.no_grad()
def continue_text(prompt, max_new=600):
    x = torch.tensor([tok.encode(prompt).ids[-BLOCK:]], device=dev)
    new = []
    while len(new) < max_new:
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x[:, -BLOCK:])[:, -1, :].float()
        nxt = sample(lg)
        x = torch.cat([x, nxt], 1)[:, -BLOCK:]
        new.append(nxt.item())
        if len(new) % 8 == 0 and len(tok.decode(new).split()) >= 128:
            break
    return tok.decode(new).split()      # new tokens only: the prompt is never included


def run_generation(prompts, tag):
    """prompts: {id: prompt_text}. Resumable via a part file keyed by tag + decoding settings."""
    part = f"{OUT}/pred_{tag}_{GEN_TAG}.json"
    pred = json.load(open(part)) if os.path.exists(part) else {}
    t0, todo = time.time(), [k for k in prompts if k not in pred]
    for n, k in enumerate(todo):
        words = continue_text(prompts[k])
        if len(words) < 128:
            raise RuntimeError(f"ID {k}: only {len(words)} words generated")
        pred[k] = {"text": " ".join(words[:128])}
        if (n + 1) % 25 == 0 or n + 1 == len(todo):
            json.dump(pred, open(part, "w"))
            print(f"{len(pred)}/{len(prompts)} done | {(time.time()-t0)/(n+1):.1f}s per prompt")
    json.dump(pred, open(part, "w"))
    assert all(len(v["text"].split()) == 128 for v in pred.values()), "not exactly 128 words"
    return pred

# ============================================================
# RUN
# ============================================================
if MODE == "test":
    ids_all = list(test.keys())
    a, b = RANGE[0], (RANGE[1] if RANGE[1] is not None else len(ids_all))
    prompts = {k: test[k]["text"] for k in ids_all[a:b]}
    pred = run_generation(prompts, f"test_{a}_{b}")
    assert not any(test[k]["text"][:60] in v["text"] for k, v in pred.items()), "prompt leaked"
    print(f"checked {len(pred)} predictions: 128 words each, no prompt text")

    if set(pred) == set(test):
        with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
            json.dump(pred, f, ensure_ascii=False)
        with zipfile.ZipFile(f"{OUT}/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        print("zip root:", zipfile.ZipFile(f"{OUT}/submission.zip").namelist(), "| IDs:", len(pred))
    else:
        print(f"PARTIAL ({len(pred)}/{len(test)} IDs): no zip written. To merge parts from several")
        print("accounts: merged = {}; [merged.update(json.load(open(p))) for p in part_files];")
        print("then assert set(merged) == set(test), write prediction.json, zip with it at the root.")

elif MODE == "val":
    rng, pairs = random.Random(1), []          # fixed seed: same pairs for every decoding setting
    for s in ["acts_val", "sc_val", "tribunals_val"]:
        c = 0
        with open(f"{RAW}/{s}.jsonl", encoding="utf-8") as f:
            for line in f:
                t = json.loads(line)["text"]
                w = t.split()
                if len(w) < 640 or any(ord(ch) > 127 for ch in t):   # test prompts are pure ASCII
                    continue
                i = rng.randint(0, len(w) - 640)
                pairs.append((s, " ".join(w[i:i + 512]), " ".join(w[i + 512:i + 640])))
                c += 1
                if c >= N_VAL_PER_SOURCE:
                    break
    prompts = {str(i): p for i, (_, p, _) in enumerate(pairs)}
    pred = run_generation(prompts, f"val_{N_VAL_PER_SOURCE}")
    refs = {str(i): {"text": r} for i, (_, _, r) in enumerate(pairs)}
    json.dump(pred, open(f"{OUT}/val_pred_{GEN_TAG}.json", "w"))
    json.dump(refs, open(f"{OUT}/val_ref.json", "w"))
    print(f"wrote val_pred_{GEN_TAG}.json and val_ref.json ({len(pairs)} pairs)")
    try:
        from rouge_score import rouge_scorer
        sc_ = rouge_scorer.RougeScorer(["rougeL"])
        for s in ["acts_val", "sc_val", "tribunals_val"]:
            ix = [i for i, (n, _, _) in enumerate(pairs) if n == s]
            if ix:
                r = np.mean([sc_.score(pairs[i][2], pred[str(i)]["text"])["rougeL"].fmeasure for i in ix])
                print(f"ROUGE-L F1 {s}: {r:.4f} (n={len(ix)})")
    except ImportError:
        print("rouge_score not installed: score val_pred / val_ref with the course evaluator instead.")
    print("Score BERTScore F1 / ROUGE-L / BLEU-4 with the COURSE evaluator for the official numbers.")
else:
    raise ValueError("MODE must be 'test' or 'val'")

loaded /kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt | step 10000 | best val 4.100625859101613
parameters: 88,210,944
checked 5 predictions: 128 words each, no prompt text
PARTIAL (5/1000 IDs): no zip written. To merge parts from several
accounts: merged = {}; [merged.update(json.load(open(p))) for p in part_files];
then assert set(merged) == set(test), write prediction.json, zip with it at the root.


In [11]:
import json, glob
p = sorted(glob.glob("/kaggle/working/submission_arch1/pred_test_0_5_*.json"))[0]
pred = json.load(open(p))
for k, v in list(pred.items())[:2]:
    print(f"--- ID {k} ---\n{v['text'][:700]}\n")
for k, v in pred.items():
    w = v["text"].split()
    print(f"ID {k}: distinct-word ratio {len(set(w))/len(w):.2f}")

--- ID 1 ---
also true that in book is not the petitioner. oversee the facts of respectfully this Court also thus only subordinate this case it is pertinent to set aside and retired. There expedient stipulations provided by the second discussion on the part of the prayer and legal practice has in his legal position. The petitioner's controversy is clearly justified in Financial 763's view ibid. In interfering with the affidavit of members, stay of the notification under Section 4 and 2 has not been confined to me depending upon normally. In exposesull v.earchridA : Karia clients of Shri V. Pharmacy (UMy) was changed in matters. Thus, it was held that when the judgment ruled that the said judgment seeks i

--- ID 2 ---
No. 6529/2009 whereasersgram drugs appeared and the complaint being prevalent by the se subsection (2) There is no ground to determine whether or not the current current financial act of the Complainant under Section jk1) (3) for around 10. Taking interim relief to Madhya

In [14]:
# ============================================================
# FINAL PIPELINE - ARCHITECTURE 1: decoding experiment -> test submission
# Self-contained (your model code is copied unchanged from the training file).
#
# MODE = "grid": for each setting in CANDIDATES, generate for the SAME held-out
#                validation pairs, save val_pred_<setting>.json (+ val_ref.json),
#                print a ROUGE-L table and append to grid_results.jsonl.
#                Score the saved files with the COURSE evaluator (BERTScore F1,
#                ROUGE-L, BLEU-4) for the official numbers.
# MODE = "test": generate for the official test prompts with the setting YOU
#                chose (GEN_CHOSEN) -> prediction.json + submission.zip
#
# Inputs attached: phase1-tokens-v40k-clean1, project-phase-1,
#                  train-model-1 notebook output (run_arch1_real/best.pt)
# ============================================================
import os, json, time, math, random, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

# ============================================================
# MODEL (your architecture, unchanged)
# ============================================================
# Published GPT-2 reference value. Confirm against your lecture; the choice is yours.
INIT_STD = 0.02


def _init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size, block_size, d_model, n_layers, n_heads, d_ff):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, block_size)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight      # tied embeddings

        self.apply(_init_weights)                        # CHANGE 1: small-std init

    def forward(self, x):
        B, T = x.shape
        assert T <= self.block_size

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        h = tok + pos

        for block in self.blocks:
            h = block(h)

        h = self.final_norm(h)
        return self.lm_head(h)


def build_model(vocab_size, block_size):
    d_model = 768
    n_layers = 8
    n_heads = 8
    d_ff = 3072

    return LegalDecoderLM(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        d_ff=d_ff,
    )




# ============================================================
# SETTINGS
# ============================================================
MODE = "grid"                  # "grid" or "test"

CKPT = "/kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt"
TOKD = "/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens"
RAW  = "/kaggle/input/datasets/achiver7/project-phase-1/data"
OUT  = "/kaggle/working/submission_arch1"
V, BLOCK, dev = 40000, 1024, "cuda"
os.makedirs(OUT, exist_ok=True)

# ---- grid mode: YOUR candidate decoding settings --------------------------
# TEMP=0 -> greedy.  TOP_K=0 -> off.  TOP_P=1.0 -> off.
# The first entry is the unmodified model distribution (your baseline row).
# SEED kept the same across candidates so the comparison is fair.
CANDIDATES = [
    # baseline: unmodified model distribution
    dict(TEMP=1.0, TOP_K=0, TOP_P=1.0, SEED=0),
    # low temperature: sharper, more conservative (greedy dropped: it loops)
    dict(TEMP=0.7, TOP_K=0, TOP_P=1.0, SEED=0),
    # top-k sampling
    dict(TEMP=0.9, TOP_K=50, TOP_P=1.0, SEED=0),
    # nucleus (top-p) sampling
    dict(TEMP=0.9, TOP_K=0, TOP_P=0.9, SEED=0),
    # combined top-k + top-p
    dict(TEMP=0.8, TOP_K=100, TOP_P=0.9, SEED=0),
]
N_VAL_PER_SOURCE = 20          # validation pairs per source (acts / sc / tribunals)

# ---- test mode: the setting YOU chose after scoring the grid ---------------
GEN_CHOSEN = None              # e.g. dict(TEMP=0.8, TOP_K=100, TOP_P=0.9, SEED=0)
RANGE = (0, 5)                 # (0, 5) = timing test; (0, None) = all 1000;
                               # split across accounts, e.g. (0, 500) and (500, None)

assert torch.cuda.is_available(), "turn on the GPU"
assert os.path.exists(CKPT), f"missing checkpoint: {CKPT}"
REQUIRED = {"TEMP", "TOP_K", "TOP_P", "SEED"}

# ============================================================
# LOAD
# ============================================================
tok = Tokenizer.from_file(f"{TOKD}/tokenizer.json")
test = json.load(open(f"{TOKD}/test.json"))

model = build_model(V, BLOCK).to(dev)
ck = torch.load(CKPT, map_location=dev)
model.load_state_dict(ck["model"])
model.eval()
CK_STEP = ck.get("step")
print(f"loaded {CKPT} | step {CK_STEP} | best val {ck.get('best')}")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

slim = f"{OUT}/model_weights_only.pt"
if not os.path.exists(slim):
    torch.save({"model": ck["model"], "step": CK_STEP}, slim)
    print("saved weights-only copy:", slim)
del ck

# ============================================================
# DECODING
# ============================================================
GEN, gen, GEN_TAG = None, None, None


def set_gen(g):
    global GEN, gen, GEN_TAG
    assert set(g) == REQUIRED, f"setting needs exactly these keys: {sorted(REQUIRED)}"
    GEN = dict(g)
    gen = torch.Generator(device=dev).manual_seed(GEN["SEED"])
    GEN_TAG = f"t{GEN['TEMP']}_k{GEN['TOP_K']}_p{GEN['TOP_P']}_s{GEN['SEED']}"


def sample(lg):
    if GEN["TEMP"] == 0:
        return lg.argmax(-1, keepdim=True)
    lg = lg / GEN["TEMP"]
    if GEN["TOP_K"]:
        v, _ = torch.topk(lg, min(GEN["TOP_K"], lg.size(-1)))
        lg[lg < v[:, [-1]]] = -float("inf")
    p = F.softmax(lg, -1)
    if GEN["TOP_P"] < 1.0:
        sp, si = torch.sort(p, descending=True)
        cum = sp.cumsum(-1)
        sp[(cum - sp) > GEN["TOP_P"]] = 0
        p = torch.zeros_like(p).scatter_(1, si, sp)
        p = p / p.sum(-1, keepdim=True)
    return torch.multinomial(p, 1, generator=gen)


@torch.no_grad()
def continue_text(prompt, max_new=600):
    x = torch.tensor([tok.encode(prompt).ids[-BLOCK:]], device=dev)
    new = []
    while len(new) < max_new:
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x[:, -BLOCK:])[:, -1, :].float()
        nxt = sample(lg)
        x = torch.cat([x, nxt], 1)[:, -BLOCK:]
        new.append(nxt.item())
        if len(new) % 8 == 0 and len(tok.decode(new).split()) >= 128:
            break
    return tok.decode(new).split()      # new tokens only: the prompt is never included


def run_generation(prompts, tag):
    """prompts: {id: prompt_text}. Resumable; part file is keyed by tag + decoding setting."""
    part = f"{OUT}/pred_{tag}_{GEN_TAG}.json"
    pred = json.load(open(part)) if os.path.exists(part) else {}
    t0, todo = time.time(), [k for k in prompts if k not in pred]
    for n, k in enumerate(todo):
        words = continue_text(prompts[k])
        if len(words) < 128:
            # do NOT crash: some decoders (greedy) can loop and produce <128 words.
            print(f"   [{GEN_TAG}] ID {k}: only {len(words)} words - padding by repetition to 128")
            base = words if words else ["the"]
            while len(words) < 128:
                words = words + base
            words = words[:128]
        pred[k] = {"text": " ".join(words[:128])}
        if (n + 1) % 25 == 0 or n + 1 == len(todo):
            json.dump(pred, open(part, "w"))
            print(f"   [{GEN_TAG}] {len(pred)}/{len(prompts)} done | {(time.time()-t0)/(n+1):.1f}s per prompt")
    json.dump(pred, open(part, "w"))
    assert all(len(v["text"].split()) == 128 for v in pred.values()), "not exactly 128 words"
    return pred

# ============================================================
# RUN
# ============================================================
if MODE == "grid":
    assert len(CANDIDATES) >= 2, "add at least one more setting to CANDIDATES"

    rng, pairs = random.Random(1), []          # fixed seed: identical pairs for every setting
    for s in ["acts_val", "sc_val", "tribunals_val"]:
        c = 0
        with open(f"{RAW}/{s}.jsonl", encoding="utf-8") as f:
            for line in f:
                t = json.loads(line)["text"]
                w = t.split()
                if len(w) < 640 or any(ord(ch) > 127 for ch in t):   # test prompts are pure ASCII
                    continue
                i = rng.randint(0, len(w) - 640)
                pairs.append((s, " ".join(w[i:i + 512]), " ".join(w[i + 512:i + 640])))
                c += 1
                if c >= N_VAL_PER_SOURCE:
                    break
    prompts = {str(i): p for i, (_, p, _) in enumerate(pairs)}
    json.dump({str(i): {"text": r} for i, (_, _, r) in enumerate(pairs)},
              open(f"{OUT}/val_ref.json", "w"))
    print(f"{len(pairs)} validation pairs ready")

    try:
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rougeL"])
    except ImportError:
        scorer = None
        print("rouge_score not installed: the table will be empty; use the course evaluator on the saved files.")

    rows = []
    for cand in CANDIDATES:
        set_gen(cand)
        print(f"--- candidate {GEN_TAG} ---")
        pred = run_generation(prompts, f"val_{N_VAL_PER_SOURCE}")
        json.dump(pred, open(f"{OUT}/val_pred_{GEN_TAG}.json", "w"))
        row = {"setting": GEN_TAG, "ckpt_step": CK_STEP, "n_per_source": N_VAL_PER_SOURCE}
        if scorer:
            for s in ["acts_val", "sc_val", "tribunals_val"]:
                ix = [i for i, (n, _, _) in enumerate(pairs) if n == s]
                row[s] = float(np.mean([scorer.score(pairs[i][2], pred[str(i)]["text"])["rougeL"].fmeasure for i in ix]))
            row["mean"] = float(np.mean([row["acts_val"], row["sc_val"], row["tribunals_val"]]))
        rows.append(row)
        with open(f"{OUT}/grid_results.jsonl", "a") as f:
            f.write(json.dumps(row) + "\n")

    print()
    print(f"{'setting':32s} {'acts':>8s} {'sc':>8s} {'tribunals':>10s} {'mean':>8s}   (ROUGE-L F1)")
    for r in rows:
        if "mean" in r:
            print(f"{r['setting']:32s} {r['acts_val']:8.4f} {r['sc_val']:8.4f} {r['tribunals_val']:10.4f} {r['mean']:8.4f}")
        else:
            print(f"{r['setting']:32s}   (not scored)")
    print("\nNext: score the val_pred_*.json files against val_ref.json with the COURSE evaluator,")
    print("pick your setting yourself, put it in GEN_CHOSEN, and set MODE = 'test'.")

elif MODE == "test":
    assert GEN_CHOSEN is not None, "set GEN_CHOSEN (the decoding setting you chose from the grid)"
    set_gen(GEN_CHOSEN)
    ids_all = list(test.keys())
    a, b = RANGE[0], (RANGE[1] if RANGE[1] is not None else len(ids_all))
    prompts = {k: test[k]["text"] for k in ids_all[a:b]}
    t0 = time.time()
    pred = run_generation(prompts, f"test_{a}_{b}")
    assert not any(test[k]["text"][:60] in v["text"] for k, v in pred.items()), "prompt leaked"
    print(f"checked {len(pred)} predictions: 128 words each, no prompt text")
    json.dump({"setting": GEN, "ckpt": CKPT, "ckpt_step": CK_STEP, "range": [a, b],
               "seconds": round(time.time() - t0)}, open(f"{OUT}/test_run_info_{a}_{b}.json", "w"))

    if set(pred) == set(test):
        with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
            json.dump(pred, f, ensure_ascii=False)
        with zipfile.ZipFile(f"{OUT}/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        print("zip root:", zipfile.ZipFile(f"{OUT}/submission.zip").namelist(), "| IDs:", len(pred))
    else:
        print(f"PARTIAL ({len(pred)}/{len(test)} IDs): no zip written. To merge parts from several")
        print("accounts: merged = {}; [merged.update(json.load(open(p))) for p in part_files];")
        print("then assert set(merged) == set(test), write prediction.json, zip with it at the root.")
else:
    raise ValueError("MODE must be 'grid' or 'test'")

loaded /kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt | step 10000 | best val 4.100625859101613
parameters: 88,210,944
60 validation pairs ready
--- candidate t1.0_k0_p1.0_s0 ---
--- candidate t0.7_k0_p1.0_s0 ---
   [t0.7_k0_p1.0_s0] 25/60 done | 2.9s per prompt
   [t0.7_k0_p1.0_s0] ID 27: only 96 words - padding by repetition to 128
   [t0.7_k0_p1.0_s0] 50/60 done | 3.0s per prompt
   [t0.7_k0_p1.0_s0] 60/60 done | 2.9s per prompt
--- candidate t0.9_k50_p1.0_s0 ---
   [t0.9_k50_p1.0_s0] 25/60 done | 2.8s per prompt
   [t0.9_k50_p1.0_s0] 50/60 done | 2.7s per prompt
   [t0.9_k50_p1.0_s0] 60/60 done | 2.7s per prompt
--- candidate t0.9_k0_p0.9_s0 ---
   [t0.9_k0_p0.9_s0] 25/60 done | 3.0s per prompt
   [t0.9_k0_p0.9_s0] 50/60 done | 2.8s per prompt
   [t0.9_k0_p0.9_s0] 60/60 done | 2.8s per prompt
--- candidate t0.8_k100_p0.9_s0 ---
   [t0.8_k100_p0.9_s0] ID 15: only 7 words - padding by repetition to 128
   [t0.8_k100_p0.9_s0] 25/60 done | 3.3s per prompt
   [t0.

In [16]:
# ============================================================
# PHASE 1 — TEST GENERATION + SUBMISSION
# Uses YOUR already-defined build_model()
# ============================================================

import os
import json
import time
import random
import zipfile

import torch
import torch.nn.functional as F
from tokenizers import Tokenizer

# ============================================================
# 1. SETTINGS
# ============================================================

V = 40000
BLOCK = 1024
dev = "cuda"

TOKD = "/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens"

CKPT = "/kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt"

OUT = "/kaggle/working/submission_arch1"

os.makedirs(OUT, exist_ok=True)

# ------------------------------------------------------------
# IMPORTANT:
# This is one of the candidate settings from YOUR grid.
# Replace these four numbers with the setting YOU selected
# after the course evaluator.
# ------------------------------------------------------------

GEN_CHOSEN = dict(
    TEMP=0.9,
    TOP_K=50,
    TOP_P=1.0,
    SEED=0
)

# First run only 5 IDs to check everything.
# After it succeeds, change to:
# RANGE = (0, None)
RANGE = (0, 5)

# ============================================================
# 2. BASIC CHECKS
# ============================================================

assert torch.cuda.is_available(), "GPU is required"
assert os.path.exists(CKPT), f"Checkpoint missing: {CKPT}"
assert os.path.exists(TOKD), f"Token directory missing: {TOKD}"
assert callable(build_model), (
    "build_model() is not defined. "
    "Run your own model-definition cell first."
)

required = {"TEMP", "TOP_K", "TOP_P", "SEED"}
assert set(GEN_CHOSEN.keys()) == required

assert GEN_CHOSEN["TEMP"] >= 0
assert GEN_CHOSEN["TOP_K"] >= 0
assert 0 < GEN_CHOSEN["TOP_P"] <= 1

print("=" * 70)
print("PHASE 1 — TEST GENERATION")
print("=" * 70)
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", CKPT)
print("Generation:", GEN_CHOSEN)
print("Range:", RANGE)

# ============================================================
# 3. TOKENIZER + TEST DATA
# ============================================================

tok = Tokenizer.from_file(
    f"{TOKD}/tokenizer.json"
)

with open(
    f"{TOKD}/test.json",
    "r",
    encoding="utf-8"
) as f:
    test = json.load(f)

print("Official test IDs:", len(test))

# ============================================================
# 4. LOAD MODEL
# ============================================================

print("\nLoading model...")

model = build_model(
    V,
    BLOCK
).to(dev)

ck = torch.load(
    CKPT,
    map_location=dev
)

model.load_state_dict(
    ck["model"]
)

model.eval()

ck_step = ck.get("step")
ck_best = ck.get("best")

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Checkpoint step:", ck_step)
print("Checkpoint best val:", ck_best)
print("Parameters:", f"{n_params:,}")
print("Parameters:", f"{n_params / 1e6:.2f}M")

assert 50e6 <= n_params <= 300e6

del ck

# ============================================================
# 5. GENERATION SETUP
# ============================================================

GEN = dict(GEN_CHOSEN)

GEN_TAG = (
    f"t{GEN['TEMP']}_"
    f"k{GEN['TOP_K']}_"
    f"p{GEN['TOP_P']}_"
    f"s{GEN['SEED']}"
)

generator = torch.Generator(
    device=dev
).manual_seed(
    GEN["SEED"]
)

print("Generation tag:", GEN_TAG)

# ============================================================
# 6. SAMPLING
# ============================================================

def sample(logits):
    """
    logits: [B, V]
    returns: [B, 1]
    """

    # Greedy
    if GEN["TEMP"] == 0:
        return logits.argmax(
            dim=-1,
            keepdim=True
        )

    logits = logits / GEN["TEMP"]

    # Top-K
    if GEN["TOP_K"] > 0:

        k = min(
            GEN["TOP_K"],
            logits.size(-1)
        )

        values, _ = torch.topk(
            logits,
            k
        )

        cutoff = values[:, [-1]]

        logits = logits.masked_fill(
            logits < cutoff,
            float("-inf")
        )

    probs = F.softmax(
        logits,
        dim=-1
    )

    # Top-P / nucleus
    if GEN["TOP_P"] < 1.0:

        sorted_probs, sorted_idx = torch.sort(
            probs,
            descending=True,
            dim=-1
        )

        cumulative = torch.cumsum(
            sorted_probs,
            dim=-1
        )

        remove = (
            (cumulative - sorted_probs)
            > GEN["TOP_P"]
        )

        sorted_probs = sorted_probs.masked_fill(
            remove,
            0.0
        )

        probs = torch.zeros_like(
            probs
        ).scatter(
            1,
            sorted_idx,
            sorted_probs
        )

        probs = probs / probs.sum(
            dim=-1,
            keepdim=True
        )

    return torch.multinomial(
        probs,
        num_samples=1,
        generator=generator
    )

# ============================================================
# 7. CONTINUATION GENERATION
# ============================================================

@torch.no_grad()
def continue_text(
    prompt,
    target_words=128,
    max_new_tokens=600
):

    prompt_ids = tok.encode(
        prompt
    ).ids

    # Keep only the model's supported context
    prompt_ids = prompt_ids[-BLOCK:]

    x = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=dev
    )

    generated = []

    for _ in range(max_new_tokens):

        model_input = x[:, -BLOCK:]

        with torch.autocast(
            "cuda",
            dtype=torch.float16
        ):

            logits = model(
                model_input
            )

        next_token = sample(
            logits[:, -1, :].float()
        )

        x = torch.cat(
            [x, next_token],
            dim=1
        )

        generated.append(
            int(next_token.item())
        )

        text = tok.decode(
            generated
        )

        words = text.split()

        if len(words) >= target_words:
            break

    return tok.decode(
        generated
    ).split()

# ============================================================
# 8. RESUMABLE GENERATION
# ============================================================

def run_generation(
    prompts,
    tag
):

    part_path = (
        f"{OUT}/pred_{tag}_{GEN_TAG}.json"
    )

    if os.path.exists(part_path):

        with open(
            part_path,
            "r",
            encoding="utf-8"
        ) as f:
            pred = json.load(f)

        print(
            "Resuming existing file:",
            part_path
        )

    else:
        pred = {}

    todo = [
        k for k in prompts
        if k not in pred
    ]

    print(
        f"Remaining: {len(todo)}/{len(prompts)}"
    )

    start = time.time()

    for count, key in enumerate(
        todo,
        start=1
    ):

        words = continue_text(
            prompts[key]
        )

        # Guarantee exactly 128 words.
        # This is the same fallback behavior as your
        # existing competition pipeline.
        if len(words) < 128:

            print(
                f"[{GEN_TAG}] ID {key}: "
                f"only {len(words)} words; "
                f"padding to 128"
            )

            base = (
                words
                if words
                else ["the"]
            )

            while len(words) < 128:
                words.extend(base)

            words = words[:128]

        else:
            words = words[:128]

        pred[key] = {
            "text": " ".join(words)
        }

        if (
            count % 5 == 0
            or count == len(todo)
        ):

            with open(
                part_path,
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(
                    pred,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

            elapsed = (
                time.time() - start
            )

            print(
                f"[{GEN_TAG}] "
                f"{len(pred)}/{len(prompts)} done | "
                f"{elapsed/max(count,1):.2f}s/prompt"
            )

    with open(
        part_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            pred,
            f,
            ensure_ascii=False,
            indent=2
        )

    # Final 128-word check
    assert all(
        len(v["text"].split()) == 128
        for v in pred.values()
    )

    return pred

# ============================================================
# 9. SELECT TEST RANGE
# ============================================================

ids_all = list(
    test.keys()
)

a = RANGE[0]

b = (
    RANGE[1]
    if RANGE[1] is not None
    else len(ids_all)
)

selected_ids = ids_all[a:b]

prompts = {
    key: test[key]["text"]
    for key in selected_ids
}

print()
print("=" * 70)
print("GENERATION")
print("=" * 70)
print(
    f"Generating IDs {a}:{b} "
    f"({len(prompts)} prompts)"
)

# ============================================================
# 10. GENERATE
# ============================================================

t0 = time.time()

pred = run_generation(
    prompts,
    f"test_{a}_{b}"
)

elapsed = (
    time.time() - t0
)

print()
print(
    f"Generation finished in "
    f"{elapsed:.2f}s"
)

print(
    f"Average: "
    f"{elapsed/max(len(pred),1):.2f}s/prompt"
)

# ============================================================
# 11. VALIDATE
# ============================================================

assert set(pred.keys()) == set(
    prompts.keys()
)

assert all(
    len(v["text"].split()) == 128
    for v in pred.values()
)

# Prompt leakage check
for key, value in pred.items():

    prompt_prefix = (
        test[key]["text"][:60]
    )

    assert (
        prompt_prefix
        not in value["text"]
    ), f"Prompt leakage: {key}"

print()
print("128-word check: PASS")
print("Prompt leakage check: PASS")
print(
    f"Generated predictions: {len(pred)}"
)

# ============================================================
# 12. TEST RUN INFO
# ============================================================

run_info = {
    "setting": GEN,
    "checkpoint": CKPT,
    "checkpoint_step": ck_step,
    "checkpoint_best_val": ck_best,
    "parameters": n_params,
    "range": [a, b],
    "count": len(pred),
    "seconds": elapsed
}

with open(
    f"{OUT}/test_run_info_{a}_{b}.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        run_info,
        f,
        indent=2
    )

# ============================================================
# 13. SAVE PARTIAL PREDICTION
# ============================================================

partial_path = (
    f"{OUT}/prediction_part_{a}_{b}.json"
)

with open(
    partial_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pred,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Saved:",
    partial_path
)

# ============================================================
# 14. CREATE FINAL SUBMISSION ONLY IF ALL IDs ARE PRESENT
# ============================================================

if set(pred.keys()) == set(test.keys()):

    prediction_path = (
        f"{OUT}/prediction.json"
    )

    with open(
        prediction_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            pred,
            f,
            ensure_ascii=False,
            indent=2
        )

    zip_path = (
        f"{OUT}/submission.zip"
    )

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        z.write(
            prediction_path,
            arcname="prediction.json"
        )

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        contents = z.namelist()

    assert contents == [
        "prediction.json"
    ]

    print()
    print("=" * 70)
    print("FINAL SUBMISSION READY")
    print("=" * 70)
    print("prediction.json:", prediction_path)
    print("submission.zip :", zip_path)
    print("ZIP contents    :", contents)
    print("IDs             :", len(pred))

else:

    print()
    print("=" * 70)
    print("PARTIAL GENERATION")
    print("=" * 70)
    print(
        f"Generated {len(pred)}/{len(test)} IDs."
    )
    print(
        "No final ZIP created."
    )
    print(
        "Use RANGE to generate the remaining IDs."
    )

PHASE 1 — TEST GENERATION
GPU: Tesla T4
Checkpoint: /kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt
Generation: {'TEMP': 0.9, 'TOP_K': 50, 'TOP_P': 1.0, 'SEED': 0}
Range: (0, 5)
Official test IDs: 1000

Loading model...
Checkpoint step: 10000
Checkpoint best val: 4.100625859101613
Parameters: 88,210,944
Parameters: 88.21M
Generation tag: t0.9_k50_p1.0_s0

GENERATION
Generating IDs 0:5 (5 prompts)
Remaining: 5/5
[t0.9_k50_p1.0_s0] 5/5 done | 2.44s/prompt

Generation finished in 12.22s
Average: 2.44s/prompt

128-word check: PASS
Prompt leakage check: PASS
Generated predictions: 5
Saved: /kaggle/working/submission_arch1/prediction_part_0_5.json

PARTIAL GENERATION
Generated 5/1000 IDs.
No final ZIP created.
Use RANGE to generate the remaining IDs.


In [17]:
# ============================================================
# FINAL SUBMISSION - ARCHITECTURE 1
# Self-contained: model code + generation + zip packaging.
# Decoding setting: TEMP=0.9, TOP_K=50, TOP_P=1.0, SEED=0
#   (the setting validated on the timing run)
# Output: /kaggle/working/submission_arch1/submission.zip
#         with prediction.json at the zip root, 1000 IDs.
# ============================================================
import os, json, time, math, zipfile
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

# ============================================================
# MODEL (unchanged)
# ============================================================
INIT_STD = 0.02


def _init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size, block_size, d_model, n_layers, n_heads, d_ff):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, block_size) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(_init_weights)

    def forward(self, x):
        B, T = x.shape
        assert T <= self.block_size
        h = self.token_emb(x) + self.pos_emb(torch.arange(T, device=x.device))
        for block in self.blocks:
            h = block(h)
        return self.lm_head(self.final_norm(h))


def build_model(vocab_size, block_size):
    return LegalDecoderLM(vocab_size, block_size, 768, 8, 8, 3072)


# ============================================================
# SETTINGS
# ============================================================
CKPT = "/kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt"
TOKD = "/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens"
OUT  = "/kaggle/working/submission_arch1"
V, BLOCK, dev = 40000, 1024, "cuda"
os.makedirs(OUT, exist_ok=True)

GEN = dict(TEMP=0.9, TOP_K=50, TOP_P=1.0, SEED=0)
GEN_TAG = f"t{GEN['TEMP']}_k{GEN['TOP_K']}_p{GEN['TOP_P']}_s{GEN['SEED']}"

assert torch.cuda.is_available(), "turn on the GPU"
assert os.path.exists(CKPT), f"missing checkpoint: {CKPT}"

# ============================================================
# LOAD
# ============================================================
tok = Tokenizer.from_file(f"{TOKD}/tokenizer.json")
test = json.load(open(f"{TOKD}/test.json"))
print(f"official test IDs: {len(test)}")

model = build_model(V, BLOCK).to(dev)
ck = torch.load(CKPT, map_location=dev)
model.load_state_dict(ck["model"])
model.eval()
print(f"loaded {CKPT} | step {ck.get('step')} | best val {ck.get('best')}")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
del ck

gen = torch.Generator(device=dev).manual_seed(GEN["SEED"])

# ============================================================
# DECODING
# ============================================================
def sample(lg):
    if GEN["TEMP"] == 0:
        return lg.argmax(-1, keepdim=True)
    lg = lg / GEN["TEMP"]
    if GEN["TOP_K"]:
        v, _ = torch.topk(lg, min(GEN["TOP_K"], lg.size(-1)))
        lg[lg < v[:, [-1]]] = -float("inf")
    p = F.softmax(lg, -1)
    if GEN["TOP_P"] < 1.0:
        sp, si = torch.sort(p, descending=True)
        cum = sp.cumsum(-1)
        sp[(cum - sp) > GEN["TOP_P"]] = 0
        p = torch.zeros_like(p).scatter_(1, si, sp)
        p = p / p.sum(-1, keepdim=True)
    return torch.multinomial(p, 1, generator=gen)


@torch.no_grad()
def continue_text(prompt, max_new=600):
    x = torch.tensor([tok.encode(prompt).ids[-BLOCK:]], device=dev)
    new = []
    while len(new) < max_new:
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x[:, -BLOCK:])[:, -1, :].float()
        nxt = sample(lg)
        x = torch.cat([x, nxt], 1)[:, -BLOCK:]
        new.append(nxt.item())
        if len(new) % 8 == 0 and len(tok.decode(new).split()) >= 128:
            break
    words = tok.decode(new).split()
    if len(words) < 128:
        base = words if words else ["the"]
        while len(words) < 128:
            words = words + base
        words = words[:128]
    return words


# ============================================================
# GENERATION (resumable, all 1000 IDs)
# ============================================================
part = f"{OUT}/pred_test_0_1000_{GEN_TAG}.json"
pred = json.load(open(part)) if os.path.exists(part) else {}
todo = [k for k in test if k not in pred]
print(f"{len(pred)} already done, {len(todo)} remaining")

t0 = time.time()
for n, k in enumerate(todo):
    words = continue_text(test[k]["text"])
    pred[k] = {"text": " ".join(words[:128])}
    if (n + 1) % 25 == 0 or n + 1 == len(todo):
        json.dump(pred, open(part, "w"))
        done = len(pred)
        elapsed = time.time() - t0
        rate = elapsed / max(n + 1, 1)
        eta = rate * (len(todo) - n - 1)
        print(f"{done}/{len(test)} done | {rate:.1f}s/prompt | ETA {eta/60:.1f} min")
json.dump(pred, open(part, "w"))

# ============================================================
# CHECKS
# ============================================================
assert all(len(v["text"].split()) == 128 for v in pred.values()), "not exactly 128 words"
assert not any(test[k]["text"][:60] in v["text"] for k, v in pred.items()), "prompt leaked"
print(f"checks PASS: {len(pred)} predictions, 128 words each, no prompt text")

# ============================================================
# FINAL PACKAGE
# ============================================================
assert set(pred) == set(test), f"missing IDs: {set(test) - set(pred)}"

with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
    json.dump(pred, f, ensure_ascii=False)
with zipfile.ZipFile(f"{OUT}/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{OUT}/prediction.json", arcname="prediction.json")

print("zip root:", zipfile.ZipFile(f"{OUT}/submission.zip").namelist())
print("IDs in zip:", len(pred))
print("final file:", f"{OUT}/submission.zip")


official test IDs: 1000
loaded /kaggle/input/notebooks/achiver7/train-model-1/run_arch1_real/best.pt | step 10000 | best val 4.100625859101613
parameters: 88,210,944
0 already done, 1000 remaining
25/1000 done | 2.7s/prompt | ETA 43.4 min
50/1000 done | 2.7s/prompt | ETA 42.0 min
75/1000 done | 2.6s/prompt | ETA 40.7 min
100/1000 done | 2.6s/prompt | ETA 39.5 min
125/1000 done | 2.6s/prompt | ETA 38.6 min
150/1000 done | 2.6s/prompt | ETA 37.3 min
175/1000 done | 2.6s/prompt | ETA 35.9 min
200/1000 done | 2.6s/prompt | ETA 35.1 min
225/1000 done | 2.6s/prompt | ETA 33.9 min
250/1000 done | 2.6s/prompt | ETA 32.8 min
275/1000 done | 2.6s/prompt | ETA 32.0 min
300/1000 done | 2.6s/prompt | ETA 30.8 min
325/1000 done | 2.6s/prompt | ETA 29.7 min
350/1000 done | 2.6s/prompt | ETA 28.6 min
375/1000 done | 2.6s/prompt | ETA 27.5 min
400/1000 done | 2.7s/prompt | ETA 26.7 min
425/1000 done | 2.7s/prompt | ETA 25.6 min
450/1000 done | 2.7s/prompt | ETA 24.5 min
475/1000 done | 2.7s/prompt | ET

In [18]:
sf

NameError: name 'sf' is not defined

In [ ]:
dcs